# Notebook 6 · LangChain vs Strands, Side by Side

### The same operations in both frameworks, so you can move between them

You now know LangChain. Strands is the other Bedrock-friendly agent framework, and the two share the same DNA: a model, some tools, a loop. Learn where they line up and where they differ, and switching becomes a lookup, not a relearn.

**How this notebook works.** The LangChain cells run live with the scripted model, so you see real output. The Strands cells are reference blocks, because Strands is a separate install and the point here is the shape of the API, not another live run. Every Strands block is labelled to run in a Strands environment.

**Standalone setup** for the runnable LangChain side.

```bash
pip install langchain langgraph langchain-aws
# the Strands side, for your own environment:
pip install strands-agents strands-agents-tools
```

In [1]:
from typing import Any, List
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain.agents import create_agent
from langchain.tools import tool


class ScriptedChatModel(BaseChatModel):
    '''Deterministic, credential-free stand-in for the LangChain side.'''
    responses: List[Any]
    idx: int = 0

    @property
    def _llm_type(self) -> str:
        return "scripted"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        reply = self.responses[min(self.idx, len(self.responses) - 1)]
        object.__setattr__(self, "idx", self.idx + 1)
        return ChatResult(generations=[ChatGeneration(message=reply)])

    def bind_tools(self, tools, **kwargs):
        return self


def show_trace(result):
    for m in result["messages"]:
        label = type(m).__name__
        if getattr(m, "tool_calls", None):
            c = m.tool_calls[0]
            print(f"{label:14} -> calls {c['name']}({c['args']})")
        else:
            print(f"{label:14} -> {m.content}")


print("setup ready")

setup ready


---
## The shared mental model

Both frameworks wrap the same loop you hand-rolled in Notebook 1. The vocabulary differs, the machine underneath does not.

```mermaid
flowchart LR
    subgraph Both frameworks
      M[model] --> D{tool needed?}
      D -->|yes| T[run tool]
      T --> M
      D -->|no| A[answer]
    end
```

The quick translation, filled in through the rest of the notebook:

| Operation | LangChain 1.0 | Strands |
|-----------|---------------|---------|
| model | `ChatBedrockConverse(model=..., region_name=...)` | `BedrockModel(model_id=..., region_name=...)` |
| create agent | `create_agent(model, tools, system_prompt=)` | `Agent(model=, tools=[], system_prompt=)` |
| run it | `agent.invoke({"messages": [...]})` | `agent("your question")` |
| final text | `result["messages"][-1].content` | `str(result)` |

---
## Rung 1: a bare agent, no tools

The baseline in both frameworks: a model wrapped in the framework's agent object, one call, one answer.

> **What runs next** a bare LangChain agent, live. The Strands equivalent follows as reference.
> **LLM concept** with no tools, both frameworks reduce to one model call in a thin wrapper.

In [2]:
model = ScriptedChatModel(responses=[AIMessage(content="I help with bookings and disruptions. What is your PNR?")])
agent = create_agent(model, tools=[], system_prompt="You are TravelMind.")
result = agent.invoke({"messages": [{"role": "user", "content": "Hi, can you help?"}]})
print("LangChain:", result["messages"][-1].content)

LangChain: I help with bookings and disruptions. What is your PNR?


> **What just happened** one human turn, one model reply. Here is the same thing in Strands. Note the ergonomic difference: Strands calls the agent like a function and reads the text with `str()`.

```python
# Strands equivalent (run in a Strands environment):
from strands import Agent
from strands.models import BedrockModel

model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0", region_name="us-east-1")
agent = Agent(model=model, system_prompt="You are TravelMind.")
result = agent("Hi, can you help?")
print("Strands:", str(result))
```

---
## Rung 2: one custom tool

Give the agent a tool. Both frameworks use an identical idea: an `@tool` decorator, and the docstring is the description the model reads.

> **What runs next** a LangChain agent that calls one tool, live, then the Strands version.
> **Language concept** in both, the docstring is the contract. Same discipline carries across frameworks.

In [3]:
@tool
def lookup_pnr(pnr: str) -> str:
    '''Return the booking status for a passenger name record (PNR).'''
    return {"JX48Q2": "BLR-DEL cancelled, Rao, Gold tier"}.get(pnr, "PNR not found")


model = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "lookup_pnr", "args": {"pnr": "JX48Q2"}, "id": "t", "type": "tool_call"}]),
    AIMessage(content="JX48Q2 is cancelled. Gold tier, so rebooking is free."),
])
agent = create_agent(model, tools=[lookup_pnr], system_prompt="You are TravelMind.")
show_trace(agent.invoke({"messages": [{"role": "user", "content": "Is JX48Q2 cancelled?"}]}))

HumanMessage   -> Is JX48Q2 cancelled?
AIMessage      -> calls lookup_pnr({'pnr': 'JX48Q2'})
ToolMessage    -> BLR-DEL cancelled, Rao, Gold tier
AIMessage      -> JX48Q2 is cancelled. Gold tier, so rebooking is free.


> **What just happened** the LangChain agent called the tool, read the record, answered. Strands defines the tool the same way and passes it in a list.

```python
# Strands equivalent (run in a Strands environment):
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def lookup_pnr(pnr: str) -> str:
    '''Return the booking status for a passenger name record (PNR).'''
    return {"JX48Q2": "BLR-DEL cancelled, Rao, Gold tier"}.get(pnr, "PNR not found")

agent = Agent(
    model=BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0", region_name="us-east-1"),
    system_prompt="You are TravelMind.",
    tools=[lookup_pnr],
)
result = agent("Is JX48Q2 cancelled?")
print(str(result))
```

> **Skeptic's corner** the tool definitions are nearly identical, which is the good news: your hard-won tool design ports across frameworks. The differences that matter are in orchestration, memory, and multi-agent, which is where the rest of the rungs live.

---
## Rung 3: several tools, and a built-in

Two things at once: multiple custom tools, and where the frameworks differ on ready-made tools. Strands ships a `strands_tools` package (a calculator, current time, HTTP, file read) that need no decorator. LangChain pulls ready-made tools from provider or community packages instead.

> **What runs next** a LangChain agent using two custom tools, live. The Strands block shows a custom tool alongside a built-in.
> **LLM concept** more tools means the descriptions carry more weight, in both frameworks, since selection is a matching problem over them.

In [4]:
@tool
def search_flights(origin: str, dest: str) -> str:
    '''Find alternate flights between two airport codes.'''
    return "AI-506 09:40, AI-812 14:15"


model = ScriptedChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "lookup_pnr", "args": {"pnr": "JX48Q2"}, "id": "a", "type": "tool_call"}]),
    AIMessage(content="", tool_calls=[{"name": "search_flights", "args": {"origin": "BLR", "dest": "DEL"}, "id": "b", "type": "tool_call"}]),
    AIMessage(content="JX48Q2 is cancelled. Alternatives: AI-506 09:40 or AI-812 14:15."),
])
agent = create_agent(model, tools=[lookup_pnr, search_flights], system_prompt="You are TravelMind.")
show_trace(agent.invoke({"messages": [{"role": "user", "content": "JX48Q2 cancelled, options?"}]}))

HumanMessage   -> JX48Q2 cancelled, options?
AIMessage      -> calls lookup_pnr({'pnr': 'JX48Q2'})
ToolMessage    -> BLR-DEL cancelled, Rao, Gold tier
AIMessage      -> calls search_flights({'origin': 'BLR', 'dest': 'DEL'})
ToolMessage    -> AI-506 09:40, AI-812 14:15
AIMessage      -> JX48Q2 is cancelled. Alternatives: AI-506 09:40 or AI-812 14:15.


> **What just happened** two tool round-trips, one grounded answer. The Strands version mixes a custom tool with a built-in from `strands_tools`.

```python
# Strands equivalent (run in a Strands environment):
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import current_time   # a built-in, no decorator needed

@tool
def search_flights(origin: str, dest: str) -> str:
    '''Find alternate flights between two airport codes.'''
    return "AI-506 09:40, AI-812 14:15"

agent = Agent(
    model=BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0", region_name="us-east-1"),
    system_prompt="You are TravelMind.",
    tools=[search_flights, current_time],
)
print(str(agent("JX48Q2 cancelled, options, and what time is it now?")))
```

---
## Rung 4: structured output

Both frameworks can return a validated object instead of prose. The mechanism differs: LangChain takes a `response_format` on the agent, Strands has a dedicated `structured_output` call.

```python
# LangChain (real model):
from pydantic import BaseModel, Field

class Disruption(BaseModel):
    pnr: str = Field(description="the PNR")
    status: str = Field(description="cancelled, delayed, or on-time")

agent = create_agent(model, tools=[...], response_format=Disruption)
result = agent.invoke({"messages": [{"role": "user", "content": "status JX48Q2?"}]})
disruption = result["structured_response"]      # a validated Disruption
```

```python
# Strands (run in a Strands environment):
from pydantic import BaseModel, Field
from strands import Agent

class Disruption(BaseModel):
    pnr: str = Field(description="the PNR")
    status: str = Field(description="cancelled, delayed, or on-time")

agent = Agent(model=..., system_prompt="You are TravelMind.")
disruption = agent.structured_output(Disruption, "status of JX48Q2?")   # a validated Disruption
```

Same destination, a typed and validated object. LangChain folds it into the agent result under `structured_response`. Strands exposes it as a separate method you call when you want structure.

---
## Rung 5: one agent as another's tool

The first step into multi-agent: wrap a specialist so a generalist can call it. Strands makes this especially direct, since an agent can be handed in as a tool. LangChain reaches the same outcome through a supervisor or by wrapping the sub-agent's call in a tool.

```python
# Strands (run in a Strands environment): an agent used directly as a tool.
from strands import Agent
from strands.models import BedrockModel

model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0", region_name="us-east-1")
flight_specialist = Agent(model=model, system_prompt="You are a flight search specialist.", tools=[search_flights])

# hand the specialist to a generalist as if it were a tool
concierge = Agent(model=model, system_prompt="You are a travel concierge.", tools=[flight_specialist])
print(str(concierge("My flight is cancelled, find options")))
```

```python
# LangChain (real model): wrap the sub-agent's invoke in a tool the parent can call.
from langchain.tools import tool

specialist = create_agent(model, tools=[search_flights], system_prompt="Flight search specialist.")

@tool
def ask_flight_specialist(request: str) -> str:
    '''Delegate a flight-search request to the specialist agent.'''
    out = specialist.invoke({"messages": [{"role": "user", "content": request}]})
    return out["messages"][-1].content

concierge = create_agent(model, tools=[ask_flight_specialist], system_prompt="Travel concierge.")
```

> **Skeptic's corner** Strands passing an agent straight in as a tool is cleaner to write. LangChain's explicit wrapper is more boilerplate but shows exactly what crosses the boundary. Neither is wrong. Pick the one whose failure modes you would rather debug.

---
## Rung 6: central coordination (supervisor)

A boss agent delegates to specialists and collects the results.

```python
# LangChain (real model):
from langgraph_supervisor import create_supervisor
from langgraph.checkpoint.memory import InMemorySaver

flight_agent = create_agent(model, tools=[search_flights], system_prompt="Flights.", name="flight_agent")
refund_agent = create_agent(model, tools=[...], system_prompt="Refunds.", name="refund_agent")

supervisor = create_supervisor(
    [flight_agent, refund_agent],
    model=model,
    prompt="Route each request to the right specialist, then summarise.",
).compile(checkpointer=InMemorySaver())
```

```python
# Strands (run in a Strands environment): the orchestrator holds specialists as tools.
concierge = Agent(
    model=model,
    system_prompt="Route each request to the right specialist, then summarise.",
    tools=[flight_specialist, refund_specialist],   # each is an Agent
)
```

LangChain names the pattern with a dedicated `create_supervisor` and an explicit compiled graph. Strands expresses the same hub with agents-as-tools. The mental model is identical: one coordinator, several specialists.

---
## Rung 7: peer handoff (swarm)

No boss. Agents hand the live conversation to each other.

```python
# LangChain (real model):
from langgraph_swarm import create_swarm, create_handoff_tool

to_booker = create_handoff_tool(agent_name="booker", description="Hand off to the booker.")
finder = create_agent(model, tools=[to_booker], system_prompt="Finder.", name="finder")
booker = create_agent(model, tools=[], system_prompt="Booker.", name="booker")
swarm = create_swarm([finder, booker], default_active_agent="finder").compile(checkpointer=InMemorySaver())
```

```python
# Strands (run in a Strands environment):
from strands.multiagent import Swarm

swarm = Swarm([finder, booker])
result = swarm("Find and book me BLR to DEL")
```

Strands offers a first-class `Swarm` that takes the agents and runs the handoff for you. LangChain builds the swarm from handoff tools and a compiled graph, which is more explicit and gives you more control over the wiring.

---
## Rung 8: a wired graph

When you need a fixed, testable path, both frameworks give you a graph builder.

```python
# LangChain / LangGraph (runs offline, no model needed for pure-Python nodes):
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)
graph.add_node("extractor", extractor)
graph.add_node("writer", writer)
graph.add_edge(START, "extractor")
graph.add_conditional_edges("extractor", route, {"writer": "writer", "ambiguity": "ambiguity"})
graph.compile()
```

```python
# Strands (run in a Strands environment):
from strands.multiagent import GraphBuilder

builder = GraphBuilder()
builder.add_node(extractor_agent, "extractor")
builder.add_node(writer_agent, "writer")
builder.set_entry_point("extractor")
builder.add_edge("extractor", "writer", condition=lambda s: s.confidence >= 0.7)
graph = builder.build()
```

Both let you pin transitions in code. LangGraph routes with a plain function returning the next node name. Strands attaches a condition to each edge. Same guarantee, slightly different wiring.

---
## The full cheat sheet

| Operation | LangChain 1.0 | Strands |
|-----------|---------------|---------|
| model | `ChatBedrockConverse(model=, region_name=)` | `BedrockModel(model_id=, region_name=)` |
| define a tool | `@tool` + docstring | `@tool` + docstring |
| built-in tools | provider or community packages | `from strands_tools import calculator, current_time, ...` |
| create an agent | `create_agent(model, tools, system_prompt=)` | `Agent(model=, tools=[], system_prompt=)` |
| run it | `agent.invoke({"messages": [...]})` | `agent("question")` |
| read the answer | `result["messages"][-1].content` | `str(result)` |
| stream | `agent.stream(..., stream_mode="updates")` | `agent.stream_async(...)` |
| structured output | `response_format=Model` then `result["structured_response"]` | `agent.structured_output(Model, "...")` |
| memory | `checkpointer=InMemorySaver()` + `thread_id` | conversation manager or session |
| approval gate | `HumanInTheLoopMiddleware` | interrupt via hooks or tool design |
| agent as a tool | wrap `invoke` in a tool, or use a supervisor | pass an `Agent` directly in `tools=[]` |
| central coordination | `langgraph_supervisor.create_supervisor` | orchestrator holding agents as tools |
| peer handoff | `langgraph_swarm.create_swarm` | `strands.multiagent.Swarm` |
| wired graph | `langgraph.StateGraph` | `strands.multiagent.GraphBuilder` |

---
## When to pick which

Neither wins outright. The honest split:

| Lean toward | If you |
|-------------|--------|
| Strands | live on Bedrock, want the leanest runtime and the fewest layers, and like agents-as-tools |
| LangChain 1.0 | need provider portability across model vendors, want middleware and a mature tracing story in LangSmith, or want the largest ecosystem |
| a raw loop | make one model call with no tools, where any framework is pure overhead |

The deepest reason to learn both: the concepts transport. Tools, the loop, memory, structured output, and multi-agent coordination are the same ideas wearing different method names. Learn one framework well and the other becomes a translation table, which is exactly what this notebook is.

> **Skeptic's corner** framework choice is rarely the thing that makes or breaks an agent. Tool design, context management, and knowing when a plain function beats an agent matter far more. Pick the framework that fits your stack, then spend your energy on the parts that actually decide quality.

---
## Series wrap-up

Six notebooks, one arc:

| # | Notebook | The one idea |
|---|----------|--------------|
| 1 | Why LangChain exists | frameworks earn their place by removing plumbing |
| 2 | Essentials | messages plus a loop is the whole machine |
| 3 | Basic | tools ground the model, and docstrings decide tool choice |
| 4 | Intermediate | memory, structured output, and guardrails turn a loop into a product |
| 5 | Advanced | graphs pin the path, multiple agents split the work |
| 6 | LangChain vs Strands | the concepts port, the method names change |

**What you can now do**

- Explain why an agent framework exists and when to skip one.
- Build agents with tools, memory, structured output, and human-approval gates.
- Drop to a graph for deterministic control, and split work across coordinated agents.
- Move between LangChain and Strands by mapping one API to the other.

**The forward-thinking view.** Model APIs keep shifting, and frameworks will keep changing method names. The invariants are the loop, the tools, and the context. Anchor your understanding there, treat the framework as a convenience, and stay ready to swap the convenience when a better one shows up.

> **The final skeptic's question** for the next agent you build, does the framework earn its complexity, or would a single model call and one plain function do the job? Ask it early. The best engineering is often the structure you chose not to add.